## Contact MinIO

In [ ]:
# %pip install boto3

In [ ]:
# %pip install python-dotenv

In [ ]:
import boto3

s3 = boto3.client(
    "s3",
    endpoint_url="http://host.docker.internal:9000",
    aws_access_key_id="minioadmin",
    aws_secret_access_key="minioadmin",
    region_name="us-east-1",
    config=boto3.session.Config(signature_version="s3v4")
)

BUCKET = "bronze"

## Reading files from MinIO

In [ ]:
# response = s3.list_buckets()

# print(response)
# print(type(response))
# print(response['Buckets'])

In [ ]:
# response = s3.list_objects_v2(Bucket=BUCKET)

# for obj in response.get("Contents", []):
#     print(obj["Key"])

In [ ]:
# import json
# import pandas as pd

# def load_latest_source_raw(source_name):
#     response = s3.list_objects_v2(Bucket=BUCKET)

#     files = [
#         obj["Key"] for obj in response.get("Contents", [])
#         if source_name in obj["Key"] and obj["Key"].endswith("data.json")
#     ]

#     if not files:
#         print(f"No files found for {source_name}")
#         return None

#     latest_file = sorted(files)[-1]

#     response = s3.get_object(Bucket=BUCKET, Key=latest_file)
#     data = json.loads(response["Body"].read().decode("utf-8"))

#     print(f"{source_name} loaded (RAW)")

#     return data
# adzuna_raw = load_latest_source_raw("adzuna")
# reed_raw = load_latest_source_raw("reed")
# arbeitnow_raw = load_latest_source_raw("arbeitnow")

In [ ]:
import os
import sys
import importlib
from datetime import datetime, timezone, timedelta

# ============================================================
# 1. ENVIRONMENT SETUP & PATH MAPPING
# ============================================================

# Ensure the project root is in sys.path so we can import 'scripts'
# Adjust the join path if your notebook is located deeper in the folder structure
project_root = os.path.abspath(os.path.join(os.getcwd(), '..')) 
if project_root not in sys.path:
    sys.path.append(project_root)

# Forcefully inject API keys into the environment BEFORE importing/reloading clients
os.environ["REED_API_KEY"] = "5f08f697-2811-4c29-b1dd-77db0440416b"
os.environ["ADZUNA_APP_ID"] = "e5976890"
os.environ["ADZUNA_APP_KEY"] = "fc8991de718748b4e89045351ad63fd3"

# ============================================================
# 2. CLIENT IMPORT & HOT RELOAD
# ============================================================

try:
    from scripts.ingestion import reed_client, adzuna_client, arbeitnow_client
    
    # Critical: Reload modules to ensure they pick up the os.environ changes injected above
    importlib.reload(reed_client)
    importlib.reload(adzuna_client)
    importlib.reload(arbeitnow_client)
    
    print("✅ Clients imported and environment variables refreshed successfully")
except ImportError as e:
    print(f"❌ Import error: {e}. Please check your project structure.")

# ============================================================
# 3. DATA COLLECTION FUNCTION
# ============================================================

def fetch_debug_data():
    """
    Fetches raw data from all three sources and handles exceptions locally 
    to prevent the entire pipeline from crashing.
    """
    adz_data, rd_data, arb_data = [], [], []

    # --- Testing Adzuna ---
    print("\n--- Testing Adzuna ---")
    try:
        # Note: Adzuna might still return 400 if the internal URL construction in adzuna_client is broken
        adz_data = adzuna_client.collect_adzuna()
        print(f"Adzuna Count: {len(adz_data)}")
    except Exception as e:
        print(f"Adzuna Failed: {e}")

    # --- Testing Reed ---
    print("\n--- Testing Reed ---")
    try:
        rd_data = reed_client.collect_reed()
        print(f"Reed Count: {len(rd_data)}")
    except Exception as e:
        print(f"Reed Failed: {e}")

    # --- Testing Arbeitnow ---
    print("\n--- Testing Arbeitnow ---")
    try:
        arb_data = arbeitnow_client.collect_arbeitnow()
        print(f"Arbeitnow Count: {len(arb_data)}")
    except Exception as e:
        print(f"Arbeitnow Failed: {e}")
        
    return adz_data, rd_data, arb_data

# ============================================================
# 4. EXECUTION
# ============================================================

# Execute the fetch and store results in variables for analysis
adzuna_raw, reed_raw, arbeitnow_raw = fetch_debug_data()

# Quick summary for the user
print("\n" + "="*30)
print(f"FINAL COLLECTION SUMMARY:")
print(f"Total Adzuna: {len(adzuna_raw)}")
print(f"Total Reed: {len(reed_raw)}")
print(f"Total Arbeitnow: {len(arbeitnow_raw)}")
print("="*30)

In [ ]:
# arbeitnow_raw

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append(r"C:\Users\Admin\OneDrive\Desktop\Projects\JobIntelligent-Data-Platform")
from scripts.processing.cleaner import clean_adzuna_data, clean_reed_data, clean_arbeitnow_data
adzuna_clean = clean_adzuna_data(adzuna_raw)
reed_clean = clean_reed_data(reed_raw)
arbeitnow_clean = clean_arbeitnow_data(arbeitnow_raw)

In [ ]:
# Display the first cleaned job from each source
print("--- First Cleaned Adzuna Job ---")
if not adzuna_clean.empty:
    display(adzuna_clean.head(1))
else:
    print("Adzuna data is empty.")

print("\n--- First Cleaned Reed Job ---")
if not reed_clean.empty:
    display(reed_clean.head(5))
else:
    print("Reed data is empty.")

print("\n--- First Cleaned Arbeitnow Job ---")
if not arbeitnow_clean.empty:
    display(arbeitnow_clean.head(1))
else:
    print("Arbeitnow data is empty.")
    

In [ ]:
# Displaying only column names for each cleaned source
print("Adzuna Columns:", adzuna_clean.columns.tolist())
print("Reed Columns:", reed_clean.columns.tolist())
print("Arbeitnow Columns:", arbeitnow_clean.columns.tolist())

## Currency Exploration

In [ ]:

print(f"{'='*20} CURRENCY STANDARDIZATION CHECK {'='*20}")

for df, name in zip([adzuna_clean, reed_clean, arbeitnow_clean], ['Adzuna', 'Reed', 'Arbeitnow']):
    if 'currency' in df.columns:
        # Check unique values and frequency
        counts = df['currency'].value_counts(dropna=False)
        print(f"\n[Source: {name}]")
        print(counts)
    else:
        print(f"\n[Source: {name}] Column 'currency' is missing.")

# Technical Depth: We look for (gbp vs GBP) or symbols (£ vs GBP)

### Code for standardizing currency 

In [ ]:
import pandas as pd
import requests
import numpy as np

def fetch_rates_to_mad():
    """
    Fetches global rates and calculates conversion factor to MAD.
    """
    try:
        # We get all rates relative to USD as a standard baseline
        url = "https://api.exchangerate-api.com/v4/latest/USD"
        response = requests.get(url)
        data = response.json()
        
        rates = data['rates']
        mad_rate_to_usd = rates.get('MAD', 10.0) # 1 USD = X MAD
        
        # Calculate: How many MAD per 1 unit of foreign currency
        # Formula: (1 / Rate_to_USD) * MAD_to_USD
        conversion_map = {curr: (1/rate) * mad_rate_to_usd for curr, rate in rates.items()}
        
        return conversion_map
    except Exception as e:
        print(f"⚠️ API Error: {e}. Using static MAD fallbacks.")
        # Fallbacks: 1 GBP ≈ 12.5 MAD | 1 EUR ≈ 10.8 MAD
        return {'GBP': 12.5, 'EUR': 10.8, 'USD': 10.1, 'MAD': 1.0}

import pandas as pd
import numpy as np

def standardize_and_convert_to_mad(df, source_name):
    """
    Standardizes currency labels and converts salaries to Moroccan Dirham (MAD).
    Handles missing values by unifying them into np.nan for analytical consistency.
    """
    df = df.copy()
    
    # Fetch latest exchange rates relative to MAD
    mad_rates = fetch_rates_to_mad()
    
    # ---------------------------------------------------------
    # 1. CURRENCY CLEANING & INFERENCE
    # ---------------------------------------------------------
    if 'currency' in df.columns:
        # Step A: Convert inconsistent string 'nulls' into real numpy NaNs
        # This prevents logic errors where 'UNKNOWN' is treated as a valid currency code
        null_variants = ['UNKNOWN', 'none', 'None', 'null', 'nan', '', ' ']
        df['currency'] = df['currency'].replace(null_variants, np.nan)
        
        # Step B: Source-based Imputation
        # If currency is missing, we infer it based on the primary market of the source
        if source_name.lower() == 'reed':
            df['currency'] = df['currency'].fillna('GBP')
        elif source_name.lower() in ['adzuna', 'arbeitnow']:
            df['currency'] = df['currency'].fillna('EUR')
        else:
            # Default to MAD if the source origin is ambiguous or local
            df['currency'] = df['currency'].fillna('MAD')

    # ---------------------------------------------------------
    # 2. CONVERSION LOGIC (ROW-LEVEL)
    # ---------------------------------------------------------
    def convert_to_mad(row, col_name):
        # Force conversion to numeric, turning errors (like 'TBC' or strings) into NaN
        val = pd.to_numeric(row[col_name], errors='coerce')
        
        # If salary is missing, we return NaN immediately to avoid unnecessary logic
        if pd.isna(val):
            return np.nan
            
        # Get currency code, defaulting to MAD if still missing
        curr = str(row.get('currency', 'MAD')).upper()
        
        # Perform conversion if the currency exists in our rates dictionary
        if curr in mad_rates:
            return round(val * mad_rates[curr], 2)
        
        # If currency code is unrecognized, return NaN to maintain data integrity
        return np.nan

    # Apply conversion to both min and max salary columns
    for salary_col in ['salary_min', 'salary_max']:
        if salary_col in df.columns:
            # Create a new standardized column with the _mad suffix
            df[f'{salary_col}_mad'] = df.apply(lambda r: convert_to_mad(r, salary_col), axis=1)
    
    # ---------------------------------------------------------
    # 3. FINAL SCHEMA ALIGNMENT
    # ---------------------------------------------------------
    # Standardize the final currency indicator:
    # Set to 'MAD' only if a numeric conversion actually took place; otherwise NaN.
    df['currency_standardized'] = df.apply(
        lambda r: 'MAD' if pd.notna(r['salary_min_mad']) else np.nan, axis=1
    )
    
    return df

# ============================================================
# EXECUTION: Standardizing our sources to Moroccan Dirham
# ============================================================
adzuna_std = standardize_and_convert_to_mad(adzuna_clean, 'Adzuna')
reed_std = standardize_and_convert_to_mad(reed_clean, 'Reed')
arbeitnow_std = standardize_and_convert_to_mad(arbeitnow_clean, 'Arbeitnow')

# Quick check on Reed data
if not reed_std.empty:
    print("\n--- Sample of Reed Salaries in MAD ---")
    display(reed_std[['job_title', 'salary_min', 'currency', 'salary_min_mad']].head(10))

## Contract Type Exploration

In [ ]:
print(f"{'='*20} CONTRACT TYPE MAPPING CHECK {'='*20}")

for df, name in zip([adzuna_clean, reed_clean, arbeitnow_clean], ['Adzuna', 'Reed', 'Arbeitnow']):
    if 'contract_type' in df.columns:
        unique_contracts = df['contract_type'].value_counts(dropna=False)
        print(f"\n[Source: {name}]")
        print(unique_contracts)
    else:
        print(f"\n[Source: {name}] Column 'contract_type' is missing.")

# Technical Depth: Adzuna and Reed often differ in naming (e.g., 'permanent' vs 'full_time')

### Code for standardizing contract types

In [ ]:
import pandas as pd
import numpy as np

def standardize_contract_types(df):
    """
    Standardizes varied contract labels into a unified taxonomy.
    """
    df = df.copy()

    # 1. Define the Canonical Taxonomy
    # Mapping various raw strings to a single standard category
   
    contract_map = {
        # --- Full-Time Category ---
        'permanent': 'Full-Time',
        'full_time': 'Full-Time',
        'full-time': 'Full-Time',
        'full time': 'Full-Time',
        'regular': 'Full-Time',
        'indefinite': 'Full-Time', # Common in European contracts (CDI)
        'cdi': 'Full-Time',        # French specific (Adzuna)

        # --- Part-Time Category ---
        'part_time': 'Part-Time',
        'part-time': 'Part-Time',
        'part time': 'Part-Time',
        'fractional': 'Part-Time',

        # --- Contractor / Freelance Category ---
        'contract': 'Contractor',
        'freelance': 'Contractor',
        'self-employed': 'Contractor',
        'external': 'Contractor',
        'independent': 'Contractor',
        'gig': 'Contractor',
        'cdd': 'Contractor',      # French temporary contract

        # --- Temporary / Seasonal Category ---
        'temporary': 'Contractor', # Often treated as contract
        'temp': 'Contractor',
        'seasonal': 'Contractor',
        'casual': 'Contractor',
        'zero-hours': 'Contractor', # Specific to UK market (Reed)

        # --- Internship / Graduate Category ---
        'intern': 'Internship',
        'internship': 'Internship',
        'placement': 'Internship',
        'apprentice': 'Internship',
        'apprenticeship': 'Internship',
        'graduate': 'Internship',
        'stage': 'Internship'     # French for Internship
    }

    if 'contract_type' in df.columns:
        # Step A: Clean strings (lowercase and remove extra spaces)
        # This handles cases like ' Full Time ' or 'Full_time'
        df['contract_type_std'] = df['contract_type'].astype(str).str.lower().str.strip().str.replace('-', '_').str.replace(' ', '_')
        
        # Step B: Apply the mapping
        # Values not found in the map will be replaced with 'Undefined'
        df['contract_type_std'] = df['contract_type_std'].map(contract_map).fillna('Undefined')
        
        # Step C: Special case for real NaNs (ensure they are 'Undefined' and not strings)
        df.loc[df['contract_type'].isna(), 'contract_type_std'] = 'Undefined'
    else:
        # If the column is completely missing, create it with 'Undefined'
        df['contract_type_std'] = 'Undefined'

    return df

# ============================================================
# EXECUTION: Applying the mapping to all sources
# ============================================================

adzuna_final = standardize_contract_types(adzuna_std)
reed_final = standardize_contract_types(reed_std)
arbeitnow_final = standardize_contract_types(arbeitnow_std)

# ============================================================
# VERIFICATION: Check the results for Adzuna (the source with data)
# ============================================================
print("--- Adzuna Contract Type: Before vs After ---")
if not adzuna_final.empty:
    comparison = adzuna_final[['contract_type', 'contract_type_std']].value_counts().head(10)
    print(comparison)

## Location Standardization Check

In [ ]:
print(f"{'='*20} LOCATION GRANULARITY CHECK {'='*20}")

for df, name in zip([adzuna_clean, reed_clean, arbeitnow_clean], ['Adzuna', 'Reed', 'Arbeitnow']):
    if 'location' in df.columns:
        # Show top 10 most frequent locations to identify patterns
        top_locations = df['location'].value_counts(dropna=False).head(10)
        print(f"\n[Source: {name}] - Top 10 Locations:")
        print(top_locations)
    else:
        print(f"\n[Source: {name}] Column 'location' is missing.")

# Technical Depth: Look for strings like 'Remote' vs structured addresses

### Code for standardizing location

In [ ]:
import pandas as pd
import numpy as np

def standardize_locations(df, source_name):
    """
    Standardizes locations and handles missing values using a 'Unknown' fallback 
    for strings and np.nan for raw data integrity.
    """
    df = df.copy()

    def parse_location(loc_str):
        # 1. Check for real Nulls (NaN/None) or placeholder strings
        null_indicators = ['nan', 'none', 'null', '', 'unknown', 'n/a']
        
        # Convert to string for checking, then handle nulls
        clean_loc = str(loc_str).lower().strip()
        
        if pd.isna(loc_str) or clean_loc in null_indicators:
            return "Unknown", "Unknown"
        
        # 2. Handle Remote Work (A specific type of location)
        if 'remote' in clean_loc or 'anywhere' in clean_loc:
            return "Remote", "Remote"
            
        # 3. Standard parsing for existing strings
        parts = [p.strip() for p in str(loc_str).split(',')]
        city = parts[0]
        country = "Unknown"

        # Mapping country based on source origin
        if source_name.lower() == 'adzuna':
            country = "France"
        elif source_name.lower() == 'reed':
            country = "United Kingdom"
        elif source_name.lower() == 'arbeitnow':
            country = "Germany"
            
        return city, country

    # Apply parsing logic to create standardized columns
    if 'location' in df.columns:
        # Extract city and country into a temp structure
        temp_data = df['location'].apply(parse_location)
        df['city_std'] = temp_data.apply(lambda x: x[0])
        df['country_std'] = temp_data.apply(lambda x: x[1])
        
        # Create final display column: "City, Country"
        # If both are Unknown, the result is "Unknown, Unknown"
        df['location_std'] = df['city_std'] + ", " + df['country_std']
    else:
        # Fallback if the column doesn't exist at all
        df['city_std'] = "Unknown"
        df['country_std'] = "Unknown"
        df['location_std'] = "Unknown, Unknown"

    return df

# ============================================================
# EXECUTION: Standardizing the 3 sources
# ============================================================
adzuna_loc_std = standardize_locations(adzuna_final, 'Adzuna')
reed_loc_std = standardize_locations(reed_final, 'Reed')
arbeitnow_loc_std = standardize_locations(arbeitnow_final, 'Arbeitnow')

# Verification for Arbeitnow (The messiest source)
print("--- Arbeitnow Location: Before vs After ---")
if not arbeitnow_loc_std.empty:
    display(arbeitnow_loc_std[['location', 'location_std']].head(10))

print("--- Reed Location: Before vs After ---")
if not reed_loc_std.empty:
    display(reed_loc_std[['location', 'location_std']].head(10))

## Temporal Analysis

In [ ]:
#  (Date formats and ranges)
print(f"{'='*20} TEMPORAL STANDARDIZATION CHECK {'='*20}")

for df, name in zip([adzuna_clean, reed_clean, arbeitnow_clean], ['Adzuna', 'Reed', 'Arbeitnow']):
    if 'posted_date' in df.columns:
        col = df['posted_date']
        
        # Checking the technical data type (Object/String vs Datetime64)
        dtype_info = col.dtype
        
        # Discovering the temporal range (How fresh is the data?)
        min_date = col.min()
        max_date = col.max()
        
        print(f"\n[Source: {name}]")
        print(f"   - Data Type: {dtype_info}")
        print(f"   - Date Range: From {min_date} to {max_date}")
        
        # Peek at the format of the first non-null record
        sample_val = col.dropna().iloc[0] if not col.dropna().empty else "N/A"
        print(f"   - Sample Format: {sample_val}")
    else:
        print(f"\n[Source: {name}] Column 'posted_date' is missing.")

# Technical Depth: If we find 'object' types, we MUST cast them using pd.to_datetime()
# during the final standardization phase to ensure sorting works correctly.

### Code for standardizing timestamps

In [ ]:
import pandas as pd
import numpy as np

# def standardize_timestamps(df, source_name):
#     """
#     Converts all date columns to a unified datetime64[ns, UTC] format.
#     Standardizes time to 00:00:00 (Midnight) for better cross-source merging.
#     """
#     df = df.copy()
    
#     # 1. Check if column exists
#     if 'posted_date' in df.columns:
#         # Step A: Robust conversion to datetime
#         # utc=True ensures all sources share the same global timeline
#         df['posted_date_std'] = pd.to_datetime(df['posted_date'], errors='coerce', utc=True)
        
#         # Step B: Individual Null Handling
#         # If a specific row is missing a date, we keep it as NaT (Not a Time)
#         # This is better for data integrity than inventing a date for every row.
#         # But if the ENTIRE source is missing (like Reed sometimes), we log it.
#         if df['posted_date_std'].isna().all():
#             print(f"⚠️ Alert: {source_name} provides no date data. Marking as NaT for later investigation.")
        
#         # Step C: Precision Normalization (Floor to Day)
#         # We align all timestamps to 00:00:00 to eliminate 'Time Noise'
#         # This makes GroupBy and Deduplication operations 100% accurate.
#         df['posted_date_std'] = df['posted_date_std'].dt.floor('D')
        
#     else:
#         # Fallback if column is missing from the schema entirely
#         print(f"❌ Error: 'posted_date' column missing in {source_name}")
#         df['posted_date_std'] = pd.NaT

#     return df
def standardize_timestamps(df, source_name):
    """
    Standardizes multiple date columns to UTC datetime at Midnight.
    Works for both Unix timestamps and standard date strings.
    """
    df = df.copy()
    
    # Define which columns should be treated as dates
    date_columns = ['posted_date', 'expires_date']
    
    for col in date_columns:
        if col in df.columns:
            # 1. Try handling as Unix Timestamps (common in Arbeitnow)
            numeric_dates = pd.to_numeric(df[col], errors='coerce')
            
            # If the column is mostly numeric, treat it as Unix seconds
            if numeric_dates.notna().sum() > (len(df) * 0.5): 
                df[f'{col}_std'] = pd.to_datetime(numeric_dates, unit='s', utc=True, errors='coerce')
            else:
                # 2. Otherwise, treat as standard date strings
                df[f'{col}_std'] = pd.to_datetime(df[col], dayfirst=True, utc=True, errors='coerce')
            
            # 3. Data Sanitization: Filter unrealistic dates
            cutoff = pd.Timestamp('2010-01-01', tz='UTC')
            df.loc[df[f'{col}_std'] < cutoff, f'{col}_std'] = pd.NaT
            
            # 4. Transform: Floor to Day (The 00:00:00 logic)
            df[f'{col}_std'] = df[f'{col}_std'].dt.floor('D')
            
    return df
# ============================================================
# EXECUTION: Standardizing our 3 sources
# ============================================================
# We use the previous standardized dataframes as input
adzuna_time_std = standardize_timestamps(adzuna_loc_std, 'Adzuna')
reed_time_std = standardize_timestamps(reed_loc_std, 'Reed')
arbeitnow_time_std = standardize_timestamps(arbeitnow_loc_std, 'Arbeitnow')

# Verification
print("\n" + "="*50)
print("--- FINAL TEMPORAL ALIGNMENT REPORT ---")
print("="*50)
for df, name in zip([adzuna_time_std, reed_time_std, arbeitnow_time_std], ['Adzuna', 'Reed', 'Arbeitnow']):
    if not df.empty:
        dtype = df['posted_date_std'].dtype
        # Get the first non-null sample if possible
        valid_samples = df['posted_date_std'].dropna()
        sample = valid_samples.iloc[0] if not valid_samples.empty else "All values are NaT"
        print(f"[{name:10}] Type: {str(dtype):15} | Sample: {sample}")

#### test 

In [1]:
def standardize_timestamps(df, source_name):
    """
    Standardizes multiple date columns to UTC datetime at Midnight.
    Works for both Unix timestamps and standard date strings.
    """
    df = df.copy()
    
    # Define which columns should be treated as dates
    date_columns = ['posted_date', 'expires_date']
    
    for col in date_columns:
        if col in df.columns:
            # 1. Try handling as Unix Timestamps (common in Arbeitnow)
            numeric_dates = pd.to_numeric(df[col], errors='coerce')
            
            # If the column is mostly numeric, treat it as Unix seconds
            if numeric_dates.notna().sum() > (len(df) * 0.5): 
                df[f'{col}_std'] = pd.to_datetime(numeric_dates, unit='s', utc=True, errors='coerce')
            else:
                # 2. Otherwise, treat as standard date strings
                df[f'{col}_std'] = pd.to_datetime(df[col], dayfirst=True, utc=True, errors='coerce')
            
            # 3. Data Sanitization: Filter unrealistic dates
            cutoff = pd.Timestamp('2010-01-01', tz='UTC')
            df.loc[df[f'{col}_std'] < cutoff, f'{col}_std'] = pd.NaT
            
            # 4. Transform: Floor to Day (The 00:00:00 logic)
            df[f'{col}_std'] = df[f'{col}_std'].dt.floor('D')
            
    return df

In [4]:
%load_ext autoreload
%autoreload 2

import sys
import os
import pandas as pd

sys.path.append(r"C:\Users\Admin\OneDrive\Desktop\Projects\JobIntelligent-Data-Platform")
from scripts.processing.cleaner import clean_adzuna_data, clean_reed_data, clean_arbeitnow_data





sys.path.append("/home/jovyan/work") 
from scripts.processing.cleaner import clean_arbeitnow_data

file_path = 'work/test/data/data.json'

if os.path.exists(file_path):
    try:
        raw_df = pd.read_json(file_path)
    except ValueError:
        raw_df = pd.read_json(file_path, lines=True)

    if not raw_df.empty:
        # إعداد العرض لرؤية جميع الأعمدة
        pd.set_option('display.max_columns', None)
        pd.set_option('display.width', 1000)

        print("="*50)
        print("📁 STEP 1: RAW DATA PREVIEW (BRONZE LAYER)")
        print("="*50)
        print(f"Original Columns: {list(raw_df.columns)}")
        print(raw_df.head(5)) 

        # 1. Cleaning
        clean_df = clean_arbeitnow_data(raw_df)
        print("="*50)
        print(f"Canonical Columns: {list(clean_df.columns)}")

        # print(clean_df.head(5))
        print("="*50)
        # 2. Transforming (Standardization)
        processed_df = standardize_timestamps(clean_df, 'Arbeitnow')

        print("\n" + "="*50)
        print("✨ STEP 2: PROCESSED DATA PREVIEW (SILVER LAYER)")
        print("="*50)
        print(f"Canonical Columns: {list(processed_df.columns)}")
        print(processed_df.head(5))

    else:
        print("⚠️ The file is empty!")
else:
    print(f"❌ File not found: {file_path}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
📁 STEP 1: RAW DATA PREVIEW (BRONZE LAYER)
Original Columns: ['slug', 'company_name', 'title', 'description', 'remote', 'url', 'tags', 'job_types', 'location', 'created_at']
                                                slug          company_name                                              title                                        description  remote                                                url                        tags                     job_types                 location          created_at
0                        data-scientist-berlin-21785                 SumUp                                     Data Scientist  <p><strong>About the team:</strong></p>\n<p>Wi...   False  https://www.arbeitnow.com/jobs/companies/sumup...                          []                            []  Berlin, Berlin, Germany 2026-04-23 20:45:23
1  senior-fullstack-engineer-operations-platform-...         

/home/jovyan/scripts/processing/cleaner.py:48: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(str(text), "html.parser")


## Identifier Integrity & Global Overlap Analysis

In [ ]:

print(f"{'='*20} ID INTEGRITY & COLLISION CHECK {'='*20}")

# Tracking IDs across all sources to detect cross-platform conflicts
id_pools = {}

for df, name in zip([adzuna_clean, reed_clean, arbeitnow_clean], ['Adzuna', 'Reed', 'Arbeitnow']):
    if 'job_id' in df.columns:
        # 1. Internal Uniqueness: Are there duplicates within the same source?
        internal_dupes = df['job_id'].duplicated().sum()
        id_pools[name] = set(df['job_id'].astype(str))
        
        print(f"\n[Source: {name}]")
        print(f"   - Internal Duplicates: {internal_dupes}")
        print(f"   - Sample ID Pattern: {df['job_id'].iloc[0] if not df.empty else 'N/A'}")

# 2. Global Overlap Check: The 'Collision' Test
print("\n" + "-"*30)
print("🔍 Cross-Source Collision Analysis:")

sources = list(id_pools.keys())
for i in range(len(sources)):
    for j in range(i + 1, len(sources)):
        src1, src2 = sources[i], sources[j]
        common = id_pools[src1].intersection(id_pools[src2])
        if common:
            print(f"   ⚠️ CONFLICT: {len(common)} IDs overlap between {src1} and {src2}!")
        else:
            print(f"   ✅ Clean: No ID overlaps between {src1} and {src2}.")

# Technical Depth: If overlaps exist, our Standardization Strategy must involve 
# 'Namespacing' (e.g., prefixing IDs with 'reed_' or 'adz_') to guarantee global uniqueness.

In [ ]:
# import pandas as pd

# def unify_job_ids(df, source_prefix):
#     """
#     Prefixes job_ids to ensure global uniqueness and traceability.
#     Example: 123 -> adz_123
#     """
#     df = df.copy()
    
#     if 'job_id' in df.columns:
#         # We convert to string first, then add the prefix
#         df['job_id'] = source_prefix.lower() + "_" + df['job_id'].astype(str)
        
#     return df

# # ============================================================
# # APPLYING NAMESPACING BEFORE MERGE
# # ============================================================
# adzuna_final = unify_job_ids(adzuna_time_std, 'adz')
# reed_final = unify_job_ids(reed_time_std, 'reed')
# arbeitnow_final = unify_job_ids(arbeitnow_time_std, 'arb')

# # Verification
# print("--- Unified ID Samples ---")
# print(f"Adzuna: {adzuna_final['job_id'].iloc[0]}")
# print(f"Reed:   {reed_final['job_id'].iloc[0]}")
# print(f"Arbeitnow: {arbeitnow_final['job_id'].iloc[0]}")

In [ ]:
# import pandas as pd
# import numpy as np
# import re

# # 1. تعريف الدوال التي تريد تجريبها
# def clean_company_names(name):
#     """Removes legal suffixes to unify company entities."""
#     if pd.isna(name): return "Unknown"
#     name = str(name).lower().strip()
#     # Remove common legal suffixes
#     name = re.sub(r'\b(llc|ltd|inc|gmbh|sa|sarl|co\.?|plc)\b', '', name)
#     return re.sub(r'\s+', ' ', name).strip().title()

# def unify_job_ids(df, source_prefix):
#     """Adds a source-based prefix to each job_id."""
#     df = df.copy()
#     if 'job_id' in df.columns:
#         df['job_id'] = source_prefix.lower() + "_" + df['job_id'].astype(str)
#     return df

# # 2. إنشاء بيانات تجريبية (Mock Data)
# data = {
#     'job_id': [101, 102, 103, 104],
#     'job_title': ['Data Engineer', 'ML Ops', 'Data Scientist', 'Backend Dev'],
#     'company_name': ['Google LLC', 'Marjane SARL', 'Amazon LTD', 'OCP Group SA']
# }

# df_test = pd.DataFrame(data)

# print("--- البيانات قبل التعديل ---")
# display(df_test)

# # 3. تجربة تنظيف أسماء الشركات
# df_test['company_name'] = df_test['company_name'].apply(clean_company_names)

# # 4. تجربة توحيد المعرفات (بافتراض أن المصدر هو Adzuna)
# df_test = unify_job_ids(df_test, 'adz')

# print("\n--- البيانات بعد التعديل (Cleaned & Unified) ---")
# display(df_test)